# Epistemic Foraging with Competing Drives


## Scenario

This experiment builds on Experiment 1 by combining it with the [T-maze](https://pymdp-rtd.readthedocs.io/en/latest/notebooks/tmaze_demo.html) example found in the pymdp repo.

Similarly to Experiment 1, we have two goal types (Food and Shelter) which provide relief from competing drives (Hunger and Stress).

This time however, each goal's true location could be in one of two places with 50% probability.

There is a 'cue' square in the corners furthest away from the potential goals. Moving towards it is a move away from the goal.

The agent is confident that if it lands on it the resulting observation will remove all uncertainty about the world state.

It will get clear information about both goal locations and so know with 100% probability which to pick.

This is acheived by having two new hidden states (true location of each goal).

They each have two possible values (Location 1 or 2).

If the agent is at any other square than the cue, it observes Null which doesn't provide any information.

If it is on the cue square, its observation includes the true locations, so it updates its state prediction to reflect that.

In [1]:
%pip install inferactively-pymdp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
from pymdp.agent import Agent
from pymdp import utils

## Hidden States

We will use a 3*3 grid here because it's enough to show the desired behaviour.

In [10]:
grid_dims = [3, 3]
n_grid_points = int(np.prod(grid_dims))
shelter_locs = [(1, 0), (2, 0)]
food_locs = [(1, 2), (2, 2)]
cue_loc = (0, 2)
max_drive = 10

print(f"Grid dimensions: {grid_dims}")
print(f"Potential food locations: {food_locs}")
print(f"Potential shelter locations: {shelter_locs}")
print(f"Cue location: {cue_loc}")
print(f"Maximum drive level: {max_drive}")

Grid dimensions: [3, 3]
Potential food locations: [(1, 2), (2, 2)]
Potential shelter locations: [(1, 0), (2, 0)]
Cue location: (0, 2)
Maximum drive level: 10


In [7]:
# Create a look-up table `loc_list` that maps linear indices to (y, x) coordinates
grid = np.arange(n_grid_points).reshape(grid_dims)
loc_list = []
for y in range(grid_dims[0]):
    for x in range(grid_dims[1]):
        loc_list.append((y, x))

In [8]:
hunger_levels = np.arange(max_drive + 1)  # 0 to max_drive, inclusive
stress_levels = np.arange(max_drive + 1) 
n_hunger = len(hunger_levels)
n_stress = len(stress_levels)

n_states = [
    n_grid_points,
    n_hunger,
    n_stress,
    2, # Food Context (2 possible locations)
    2 # Shelter Context (2 possible locations)
]

n_obs = [
    n_grid_points,
    n_hunger,
    n_stress,
    3, # Food Cue observation (Null, Hint0, Hint1)
    3  # Shelter Cue observation (Null, Hint0, Hint1)
]

## A Matrix: Observation -> State

The A matrix is similar to the previous experiments in that location, hunger and stress are perfect observation, identity matrices.

This time we have also have the Food and Shelter Context hidden states, so the A matrix encodes that
- If you aren't on Cue, expect to observe Null which provides no information about the contexts.
- If you are on Cue, expect to observe confident context predictions.

In effect, visiting the Cue reveals the true state of the world which is otherwise masked.

In [11]:
A3 = utils.obj_array_zeros([[n] + n_states for n in n_obs])

# Fill A3
for loc_idx in range(n_grid_points):
    loc_yx = loc_list[loc_idx]
    
    for h in range(n_hunger):
        for s in range(n_stress):
            for f_ctx in range(2):
                for s_ctx in range(2):
                    # A3[0] Location: Identity
                    A3[0][loc_idx, loc_idx, h, s, f_ctx, s_ctx] = 1.0
                    
                    # A3[1] Hunger Sensor:
                    A3[1][h, loc_idx, h, s, f_ctx, s_ctx] = 1.0
                    
                    # A3[2] Stress Sensor:
                    A3[2][s, loc_idx, h, s, f_ctx, s_ctx] = 1.0
                        
                    # A3[3] Food Cue Obs - reveals FoodCtx at cues AND at food candidate locations
                    # A3[4] Shelter Cue Obs - reveals ShelterCtx at cues AND at shelter candidate locations
                    # This allows learning from visiting false reward locations
                    
                    # Check if at a cue (reveals both contexts)
                    at_cue = (loc_yx == cue_loc)
                    # Check if at a food candidate (reveals food context)
                    at_food_candidate = (loc_yx in food_locs)
                    # Check if at a shelter candidate (reveals shelter context)
                    at_shelter_candidate = (loc_yx in shelter_locs)
                    
                    # Food Cue observation
                    if at_cue or at_food_candidate:
                        A3[3][f_ctx + 1, loc_idx, h, s, f_ctx, s_ctx] = 1.0
                    else:
                        A3[3][0, loc_idx, h, s, f_ctx, s_ctx] = 1.0  # Null
                    
                    # Shelter Cue observation
                    if at_cue or at_shelter_candidate:
                        A3[4][s_ctx + 1, loc_idx, h, s, f_ctx, s_ctx] = 1.0
                    else:
                        A3[4][0, loc_idx, h, s, f_ctx, s_ctx] = 1.0  # Null

## B Matrix: State -> Action -> State


The B matrix has pre encoded that

- There's a 100% correlation between the two goal location states (0 or 1) and which location transitions lead to relief from hunger / stress. That implies the more confidently you know those state vars, the more confident you are of the locations that successful policies will visit.

- Moving into a state where the location is not the true location but *is* in the candidate list (i.e. is the wrong choice) *will* transition to *maximum* hunger / stress on the next timestep. This adds an incentive to avoid gambling as there is an actual penalty, dissuading an agent from travelling through both candidates if it was a shorter path than info. 

- Your hunger state tansition from t to t+1 is a function of
1.) Your current hunger
2.) Your location
3.) The true goal location
i.e. If not a candidate, current_value +1 / If true goal, =0 / If candidate but not true goal, =max

This means we need a 3D B matrix for each of the hunger and state single-possible-action transitions.

We need a policy length of 4 in order to let the agent to move between any two points (max 3 manhatten for travel time, +1 timestep for imagining the result eat / rest when you get there).

Example transitions -
- 'If I land on the real food pile at time t, I will transition to minimum hunger at t+1'
- 'If I land on the fake food pile at time t, I will transition to max hunger at t+1'
- 'If I land on any square that isn't the food pile at time t, I will get an (increasingly disliked) hunger increment of +1 at t+1'
- 'At the start, landing on a candidate square may reset *or* max out my hunger depending upon the hidden goal var which we are ambiguous about. Any policy that passes through them has equal chance of win or catastrophe, I can't confidently score it.'
- 'If I land on the cue square, I observe the goal var for certain so I am no longer ambiguous about the state of the world and know how landing on a candidate will turn out so *can* confidently score that policy'
- 'I might end up in a more hungry / stressed state by visiting the info, but crucially I won't end up as bad as if I had gambled and lost'
- 'If both policies ultimately end up in max discomfort, who cares which I choose?'

You could tweak the parameters to balance risk avoidance and get it to gamble over info seeking, depending on how much it hates small amounts of hunger and stress.